# Quick benchmark — pick a variant, run a few epochs, read the wall clock

Throughput and time-per-epoch for one PVT v2 size on **this** machine, so a
90/150/300-epoch budget can be planned from a measurement instead of an
estimate. Nothing here is a real run: no W&B, checkpoints go to a scratch
directory, and the epoch can be capped to a number of batches.

It needs an Arrow snapshot, but not the full 160 GB one. On a new box build a
fraction (`python download_data.py --out <DATA_DIR> --fraction 0.25`, the
validation split always in full) or carve ~20k images out of an existing
snapshot and copy the 2-3 GB over (`python download_data.py --from-snapshot
/data/imagenet_arrow --out <DATA_DIR> --n-train 20000 --n-val 2000`). See the
page-cache caveat below before reading anything but GPU throughput off a subset.

Edit the CONFIG cell, run all.

In [ ]:
import importlib, pathlib, subprocess, sys, time

REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / "pvt_moe").is_dir():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "pvt_moe").is_dir(), f"pvt_moe not found from {pathlib.Path.cwd()}"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
print(f"torch {torch.__version__} | CUDA {torch.version.cuda}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} (sm_{p.major}{p.minor}, {p.total_memory / 1024**3:.1f} GiB)")
else:
    print("no GPU visible — the numbers below will not mean anything")

## CONFIG — the only cell to edit

`LIMIT_TRAIN_BATCHES = None` times full epochs (45–85 min each for B1 on a
5070). `200` times 200 batches per epoch and extrapolates to the full epoch
from the measured images/s — enough for a throughput check in minutes.

## Subsets and the page-cache caveat

A 2-3 GB subset (a `--fraction` build or a `--from-snapshot` carve) fits
entirely in the OS page cache after one pass over it, so from the second epoch
on the disk is never read and the dataloader looks faster than it ever is on
the full 160 GB snapshot. That is FINE for comparing GPU compute across
machines - it removes the disk as a variable - but subset numbers must NOT be
used to re-answer `NUM_WORKERS` (the config's `num_workers`) or to estimate
real epoch times; both depend on the disk that the subset hides.

Baseline to compare a new machine against - RTX 5090, 24 cores, **full**
snapshot, batch 128, B1 dense:

| measurement | img/s | note |
|---|---|---|
| dataloader only, 4 workers | 1,411 | |
| dataloader only, 8 workers | 2,712 | |
| dataloader only, 12 workers | 3,800 | |
| dataloader only, 16 workers | 4,098 | plateau |
| dataloader only, 20 workers | 4,030 | |
| dataloader only, 24 workers | 3,952 | |
| real training | 2,287 | 9.3 min/epoch, GPU-bound |

Dataloader-only throughput plateaus at 16 workers (4,098 img/s); real training
runs at 2,287 img/s, so the GPU, not the loader, is the bottleneck on that box.


In [ ]:
import os
from pvt_moe import default_config, merge_config, validate_config
from pvt_moe.cli import describe

DATA_DIR            = "/data/imagenet_arrow"   # per-machine: the Arrow snapshot directory (dataset_dict.json inside)
VARIANT             = "b1"     # "b0" .. "b5"
EPOCHS              = 5        # epochs to time
LIMIT_TRAIN_BATCHES = None     # None = full epochs; e.g. 200 = 200 batches/epoch
LIMIT_VAL_BATCHES   = 50       # keep validation short; None = full val set
BATCH_SIZE          = 128      # MICRO-batch (B2: try 64; B0: 512)
EFFECTIVE_BATCH     = 1024     # accumulation is derived
NUM_WORKERS         = 8
USE_MOE             = True     # False = dense baseline
BACKEND             = "tutel"  # "native" if tutel is not installed
SCRATCH_DIR         = "bench_runs"   # checkpoints/logs for these throwaway runs
if not os.path.isfile(os.path.join(DATA_DIR, "dataset_dict.json")):
    raise SystemExit(
        f"no Arrow snapshot at {DATA_DIR} (no dataset_dict.json inside). Set DATA_DIR above, or build one:\n"
        "  python download_data.py --out <DATA_DIR> --fraction 0.25\n"
        "  python download_data.py --from-snapshot /data/imagenet_arrow --out <DATA_DIR> --n-train 20000 --n-val 2000")

overrides = {
    "recipe": "scratch",
    "epochs": EPOCHS,
    "limit_train_batches": LIMIT_TRAIN_BATCHES,
    "limit_val_batches": LIMIT_VAL_BATCHES,
    "batch_size": BATCH_SIZE,
    "effective_batch_size": EFFECTIVE_BATCH,
    "num_workers": NUM_WORKERS,
    "use_wandb": False,
    "use_tensorboard": False,
    "checkpoint_root": SCRATCH_DIR,
    "log_root": SCRATCH_DIR,
    "model": {"variant": VARIANT,
              "moe": {"backend": BACKEND},
              "ablation": {"use_moe": USE_MOE}},
}
overrides["dataset"] = {"arrow_dirs": {"imagenet-1k": DATA_DIR}}

cfg = validate_config(merge_config(default_config(), overrides))
print(describe(cfg))
print(f"\nequivalent CLI: python train.py --variant {VARIANT} --epochs {EPOCHS} "
      f"--batch-size {BATCH_SIZE} --effective-batch-size {EFFECTIVE_BATCH} --no-wandb"
      + (f" --set limit_train_batches={LIMIT_TRAIN_BATCHES}" if LIMIT_TRAIN_BATCHES else "")
      + (f" --set limit_val_batches={LIMIT_VAL_BATCHES}" if LIMIT_VAL_BATCHES else ""))

## Run

In [ ]:
import pytorch_lightning as pl
from pvt_moe.data import build_dataloaders
from pvt_moe.engine import LitClassifier, build_trainer, setup_environment


class Stopwatch(pl.Callback):
    """Per-epoch wall clock and images/s for the training loop."""

    def __init__(self, micro_batch):
        self.micro = micro_batch
        self.rows = []

    def on_train_epoch_start(self, trainer, pl_module):
        self._t0 = time.time(); self._n = 0
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        self._n += 1

    def on_train_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        secs = time.time() - self._t0
        peak = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
        self.rows.append({"epoch": trainer.current_epoch, "batches": self._n,
                          "train_s": secs, "img_s": self._n * self.micro / secs,
                          "peak_gib": peak})


setup_environment(cfg)
train_loader, val_loader = build_dataloaders(cfg)
batches_per_full_epoch = len(train_loader)
model = LitClassifier(cfg)
watch = Stopwatch(cfg["batch_size"])
trainer = build_trainer(cfg, extra_callbacks=[watch])
t_all = time.time()
trainer.fit(model, train_loader, val_loader)
print(f"total wall clock incl. validation: {(time.time() - t_all) / 60:.1f} min")

## Result

In [ ]:
rows = watch.rows[1:] or watch.rows          # drop the warm-up epoch when we have more
img_s = sum(r["img_s"] for r in rows) / len(rows)
full_epoch_min = batches_per_full_epoch * cfg["batch_size"] / img_s / 60
print(f"{'epoch':>5} {'batches':>8} {'train s':>9} {'img/s':>8} {'peak GiB':>9}")
for r in watch.rows:
    print(f"{r['epoch']:>5} {r['batches']:>8} {r['train_s']:>9.0f} {r['img_s']:>8.0f} {r['peak_gib']:>9.2f}")
print(f"\nvariant {VARIANT} | micro-batch {cfg['batch_size']} x {cfg['accumulate_grad_batches']} accum")
print(f"steady-state throughput : {img_s:.0f} img/s (train loop only)")
print(f"one FULL epoch          : {full_epoch_min:.0f} min "
      f"({batches_per_full_epoch} batches{' — extrapolated' if LIMIT_TRAIN_BATCHES else ' — measured'})")
for budget in (90, 150, 300):
    print(f"  {budget:>3} epochs             : {full_epoch_min * budget / 60 / 24:.1f} days (+ validation)")